In [1]:
# from google.colab import drive
import torch
import sys
# drive.mount('/content/gdrive', force_remount=True)


#base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/Final Project"
base_dir = "/content/gdrive/MyDrive/Final Project"

sys.path.append(base_dir)

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/davidheimowitz/.pyenv/versions/3.12.5/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/davidheimowitz/.pyenv/versions/3.12.5/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/davidheimowitz/.pyenv/versions/3.12.5/lib/python3.12/site-packages/ipykernel/kernelapp.py

Device set to mps


In [2]:
import importlib
import Models.GPT_Model as GPT_Model
import Datasets.DataLoader as DataLoader_Lib

importlib.reload(GPT_Model)
importlib.reload(DataLoader_Lib)

from Models.GPT_Model import GPT2_Lag, GPTConfig
from Datasets.DataLoader import TinyShakespeareDataLoader, TinyStoriesDataLoader

/Users/davidheimowitz/.pyenv/versions/3.12.5/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
importlib.reload(GPT_Model)
B = 4 
block_size = 128

loader = TinyStoriesDataLoader(max_length= block_size, batch_size=B)
train_loader, val_loader = loader.get_data()


Map: 100%|██████████| 10000/10000 [00:01<00:00, 7383.09 examples/s]


In [4]:
Config = GPTConfig(num_heads = 12,
  num_layers = 12,
  vocab_size = 50257,
  embedding_dim = 768,
  block_size = block_size,
  lag_behind = 1,
  dropout = .1,
  pad_token_id=loader.pad_token())

model = GPT2_Lag(Config, device)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [5]:
batch = next(iter(train_loader))
print(batch.keys())

dict_keys(['input_ids'])


In [6]:
@torch.inference_mode()
def estimate_loss_and_perplexity(model, loader, device, eval_iters=10):
    model.eval()

    total_loss = 0.0
    val_iter = iter(loader)

    for _ in range(eval_iters):
        try:
            batch = next(val_iter)
        except StopIteration:
            val_iter = iter(loader)
            batch = next(val_iter)

        input_ids = batch["input_ids"].to(device)
        labels = input_ids

        _, _, loss = model(input_ids, labels)
        total_loss += loss

    avg_loss = total_loss.item() / eval_iters
    perplexity = torch.exp(torch.tensor(avg_loss)).item()

    model.train()
    return avg_loss, perplexity

def train_loop(model, optimizer, device, train_loader, val_loader, num_steps_train, num_steps_val):
    print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    train_iter = iter(train_loader)  # create iterator once

    for step in range(num_steps_train):
        if step % 100 == 0:
            val_loss, val_ppl = estimate_loss_and_perplexity(model, val_loader, device, num_steps_val)
            print(f"Step {step:4d} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.4f}")

        # Restart iterator if all data has been seen
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        labels    = input_ids

        _, _, loss = model(input_ids, labels)
        loss.backward()
        optimizer.step()

        if step % 20 == 0:
            print(f"Step {step:4d} train loss {loss.item():.4f}")
    val_loss, val_ppl = estimate_loss_and_perplexity(model, val_loader, device, num_steps_val)
    print(f"Step {step:4d} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.4f}")

In [8]:
num_steps_train = 1000
num_steps_val = 10
train_loop(model, optimizer, device, train_loader, val_loader,num_steps_train, num_steps_val)

Number of trainable parameters: 123,751,680
Step    0 | Val Loss: 5.3381 | Val PPL: 208.1243
Step    0 train loss 5.5538
Step   20 train loss 5.5237
Step   40 train loss 6.0837
Step   60 train loss 5.7220
Step   80 train loss 6.0090
Step  100 | Val Loss: 5.2705 | Val PPL: 194.5189
Step  100 train loss 5.4095
Step  120 train loss 6.3775
Step  140 train loss 5.4404
Step  160 train loss 5.4070
Step  180 train loss 5.1797
Step  200 | Val Loss: 5.0787 | Val PPL: 160.5634
Step  200 train loss 5.0529
Step  220 train loss 5.2529
Step  240 train loss 4.8943
Step  260 train loss 5.3103
Step  280 train loss 5.0610
Step  300 | Val Loss: 4.8513 | Val PPL: 127.9036
Step  300 train loss 5.1360
Step  320 train loss 4.8606
Step  340 train loss 4.8858
Step  360 train loss 5.2771
Step  380 train loss 4.6710
Step  400 | Val Loss: 4.6037 | Val PPL: 99.8568
Step  400 train loss 5.0542
Step  420 train loss 5.3017
Step  440 train loss 4.5251
Step  460 train loss 4.7737
Step  480 train loss 4.9436
Step  500 | 